In [5]:
import os
import sys
if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))
# os.environ["PYTHONPATH"] = '/opt/venv/lib/python3.11/site-packages'

import gc
import logging
import multiprocessing
import pickle
import time
import datetime
from functools import partial
from pathlib import Path
from typing import Any, Literal, Union

import dgl
import numpy as np
import pandas as pd
import pysmiles
import torch
from datasets import Dataset, IterableDataset, disable_caching, load_dataset, load_from_disk
from dgl.dataloading import GraphDataLoader
from dgl.heterograph import DGLGraph

from rdkit import Chem
from tqdm.auto import tqdm
import lancedb
import pyarrow as pa

from molr.data_processing import mol_to_dgl, networkx_to_dgl
from molr.model import GNN
from molr.interface import MolRSmilesEmbedder
logging.getLogger('pysmiles').setLevel(logging.CRITICAL)

from multiprocess import set_start_method
try:
    set_start_method("spawn")
except RuntimeError as e:
    print(e)

disable_caching()  # disables caching for datasets

context has already been set


In [2]:
MODEL_PATH = Path('../saved/tag_1024')
DATA_PATH = Path("../../data/CS2/train.csv")

embedder = MolRSmilesEmbedder(MODEL_PATH)
dataset = load_dataset('csv', data_files=str(DATA_PATH), split='train')

In [ ]:
emb_dataset_head = dataset.select(range(1)).map(embedder.process_hf_dataset, batched=False, batch_size=128)
emb_dataset = dataset.map(embedder.process_hf_dataset, batched=True, batch_size=128)

In [12]:
time.strftime("%Y-%m-%d %H:%M:%S", time.time())

TypeError: Tuple or struct_time argument required

In [ ]:
emb_dataset = load_from_disk(DATA_PATH.parent / 'all_embeddings')
dimension = len(emb_dataset[0]['vector'])
print(f"embedding dimension: {dimension}")

In [7]:
uri = "../data/lancedb_demo"
db = lancedb.connect(uri)


In [8]:
def create_schema_from_dataset(
    dataset: Dataset, dimension: int, only_fields: Union[list[str], None] = None):
    schema = dataset.features.arrow_schema.remove_metadata()

    # replace the vector field if it already exists with a fixed size list
    idx = schema.get_field_index("vector")
    if idx != -1:
        schema = schema.remove(schema.get_field_index("vector"))
    schema = schema.insert(0, pa.field("vector", pa.list_(pa.float32(), list_size=dimension)))

    # remove all fields that are not in the only_fields list
    all_fields = dataset.features.keys()
    if only_fields is not None:
        for field in all_fields:
            if field not in only_fields:
                schema = schema.remove(schema.get_field_index(field))
    return schema


In [9]:
schema = create_schema_from_dataset(emb_dataset_head, dimension=1024)
db.drop_table("chembl", ignore_missing=True)
tbl = db.create_table("chembl", schema=schema)

In [ ]:
start = time.time()
tbl.add(emb_dataset)
end = time.time()
import_time = end - start
print(f"Time to import data into db: {import_time // 60:.2f} min {import_time % 60:.2f} seconds")

In [ ]:
start = time.time()
tbl.create_index(
    num_partitions=int(np.sqrt(len(emb_dataset))),  # number of centroids
    num_sub_vectors=64,                             # divisor of the vector dimension
    accelerator="cuda"                              # build kNN index on GPU
)
end = time.time()
indexing_time = end - start
print(f"Indexing time: {import_time // 60:.2f} min {indexing_time:.2f} seconds")

In [31]:
def extract_centroids(dataset):
    index_name = dataset.list_indices()[0]["name"]
    index_stats = dataset.index_statistics(index_name)
    n_indices = len(index_stats["indices"])
    if n_indices > 1:
        raise ValueError(f"Only one index is supported, got {n_indices}")
    n_dim = index_stats["indices"][0]['sub_index']["dimension"]
    centroids_values = pa.array(
        [x for c in index_stats['indices'][0]["centroids"] for x in c],
        pa.float32()
    )
    return pa.FixedSizeListArray.from_arrays(centroids_values, n_dim)
    # return pa.Table.from_arrays([column], ["centroids"])

In [32]:
centroids = extract_centroids(tbl._dataset)


In [ ]:
centroids[0].values.to_numpy() + centroids[1].values.to_numpy()

In [57]:
db.drop_table("centroids", ignore_missing=True)
tbl_centr = db.create_table(
    "centroids", data=pa.Table.from_arrays([centroids, pa.array(range(len(centroids)))], ["vector", "cluster_id"]))

In [ ]:
tbl_centr.create_scalar_index("cluster_id", replace=True)
tbl_centr.create_index(
    num_partitions=int(np.sqrt(len(centroids))),    # number of centroids
    num_sub_vectors=1,                              # divisor of the vector dimension
    accelerator="cuda"                              # build kNN index on GPU
)
tbl_centr.search(centroids[0].values.to_numpy() + centroids[1].values.to_numpy()).where("cluster_id > 10", prefilter=True).to_pandas()

In [12]:
def test_search_speed(table, dataset, batch_sizes=[16, 32, 64, 128], num_retries=5):
    times = []
    accuracies_top1 = []
    accuracies_top5 = []
    for batch_size in tqdm(batch_sizes, leave=False, desc='batch_size', position=0):
        times_per_run = []
        accuracies_top1_per_run = []
        accuracies_top5_per_run = []
        for _ in range(num_retries):
            random_indices = np.random.randint(0, len(dataset) - 1, batch_size)
            random_smiles = [dd for dd in dataset.select(random_indices)['smiles']]
            data = [vec for vec in dataset.select(random_indices)['vector']]
            start = time.time()
            res = [table.search(vec).limit(5).select(["smiles", "main_reward"]).to_list() for vec in data]
            end = time.time()
            times_per_run.append(end - start)
            retrieved_smiles_top1 = [per_query[0]['smiles'] for per_query in res]
            retrieved_smiles_top5 = [dd['smiles'] for per_query in res for dd in per_query]
            accuracies_top1_per_run.append(len(set(retrieved_smiles_top1).intersection(set(random_smiles))) / batch_size)
            accuracies_top5_per_run.append(len(set(retrieved_smiles_top5).intersection(set(random_smiles))) / batch_size)
        times.append(np.mean(times_per_run))
        accuracies_top1.append(np.mean(accuracies_top1_per_run))
        accuracies_top5.append(np.mean(accuracies_top5_per_run))
    result = pd.DataFrame({'batch_size': batch_sizes, 'time': times, 'accuracy_top1': accuracies_top1, 'accuracy_top5': accuracies_top5})
    result['time_per_mol'] = result['time'] / result['batch_size']
    return result

In [ ]:
results = test_search_speed(tbl, emb_dataset, batch_sizes=[16, 32, 64, 128], num_retries=5)

In [ ]:
results.plot(x='batch_size', y='time_per_mol', title='Search time per molecule', ylabel='time (s)', xlabel='batch size')

In [ ]:
results.plot(x='batch_size', y='accuracy_top1', title='Top 1 recall', ylabel='recall', xlabel='batch size')